# Notebook 27 — Revised forward-compute protocol

**Author:** Sunita Sahu · **Revision items:** A5 (cost model), A7 (ViLT), A9 (token convention)

The published Table 3 priced each model at a hand-chosen 'default native configuration'. Those choices were not comparable: Pixtral at 1024 × 1024 (4,096 tokens), Qwen2-VL at 448 × 448 (256 tokens), LLaVA-NeXT at a 576-token view its released processor never produces, Idefics2 without the image splitting its released processor enables, DeepSeek-VL2 at 729 tokens although its adaptor compresses each tile to 196. This notebook prices every model on the **same input image** through its **released preprocessing**, with a cost model checked against an op-level counter.

---
## Setup

In [1]:
import os, sys, math, json, time
from pathlib import Path
import numpy as np
import pandas as pd

os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
NOTEBOOK_DIR = Path.cwd()
CODE_ACTIVE = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
sys.path.insert(0, str(CODE_ACTIVE))
MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'

from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
from vlm_audit.parameter_layouts import ROSTER_TABLE_NAMES, load_parameter_shapes
from vlm_audit.token_accounting import TokenAccount, account_visual_tokens
from vlm_audit.tensor_cost_model import forward_cost_revised
from vlm_audit.flop_counter_validation import VALIDATION_SPECS, validate_model, measure_elementwise_share
from vlm_audit.analytic_compute import fisher_interval
from vlm_audit.correlation_grid import spearman_permutation_test
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 80)

TABLE_ORDER = list(ROSTER_TABLE_NAMES)
parameter_shapes = {name: load_parameter_shapes(name) for name in TABLE_ORDER}
components = pd.read_csv(MEASUREMENT_DIR / 'nb26_parameter_components_v2.csv').set_index('model')
REFERENCE_IMAGE, SMALL_IMAGE = (1024, 1024), (448, 448)

<home>\AppData\Local\Temp\claude\<session>\0edba254-4773-49bc-b009-31568158ee2c\scratchpad\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1. Visual tokens from the released preprocessing (revised Table 4)

For each model: tokens entering the vision encoder (summed over crops/tiles), tokens leaving the fusion module, and tokens the image adds to the language-model sequence (placeholders + layout tokens) or that cross-attention layers read. The dynamic maximum is the largest count over a sweep of 36 image shapes (long side 448–4,096 px, aspect ratios 1:4 to 4:1).

In [2]:
accounts = {size: {name: account_visual_tokens(name, *size) for name in TABLE_ORDER} for size in (SMALL_IMAGE, REFERENCE_IMAGE)}

def boundary_tokens(account):
    """Fusion-boundary bandwidth: tokens the language side receives (inserted or cross-attended)."""
    return account.language_visual_tokens or account.cross_attended_tokens or account.fusion_output_tokens

SWEEP = sorted({(round(long_side * ratio_height), round(long_side * ratio_width))
                for long_side in (448, 1024, 2048, 4096)
                for ratio_height, ratio_width in ((1, 1), (1, .5), (.5, 1), (1, 1/3), (1/3, 1), (1, .25), (.25, 1), (1, .75), (.75, 1))})
maximum = {}
for name in TABLE_ORDER:
    best = max((account_visual_tokens(name, height, width) for height, width in SWEEP), key=boundary_tokens)
    maximum[name] = best

PUBLISHED_TABLE4 = {'CLIP-ViT-L/14': (1, 1), 'SigLIP-L/16': (1, 1), 'SigLIP2-So400m/14': (1, 1), 'BLIP-large': (1, 1),
    'BLIP-2-OPT-2.7B': (32, 32), 'InstructBLIP-Vicuna-7B': (32, 32), 'IDEFICS-9B': (64, 64), 'Idefics2-8B': (64, 64),
    'ViLT-B/32': (144, 144), 'Qwen2-VL-7B': (256, 1024), 'Qwen2.5-VL-7B': (256, 1024), 'InternVL3-8B': (256, 3072),
    'LLaVA-1.5-7B': (576, 576), 'LLaVA-NeXT-7B': (576, 2880), 'Molmo-7B-D': (576, 2560), 'Janus-Pro-7B': (576, 576),
    'DeepSeek-VL2-Small': (729, 729), 'Chameleon-7B': (1024, 1024), 'Llama-3.2-11B-Vision': (1601, 1601), 'Pixtral-12B': (4096, 4096)}
table4_rows = []
for name in TABLE_ORDER:
    small, reference, largest = accounts[SMALL_IMAGE][name], accounts[REFERENCE_IMAGE][name], maximum[name]
    published_default, published_dynamic_max = PUBLISHED_TABLE4.get(name, (None, None))   # None: added under D9
    table4_rows.append({'model': name, 'published_default': published_default, 'published_dynamic_max': published_dynamic_max,
                        'tokens_448': boundary_tokens(small), 'tokens_1024': boundary_tokens(reference),
                        'tokens_max': boundary_tokens(largest), 'max_at': f'{largest.image_height}x{largest.image_width}',
                        'encoder_tokens_1024': reference.encoder_tokens, 'encoder_passes_1024': len(reference.encoder_sequences),
                        'details_1024': '; '.join(f'{key}={value}' for key, value in reference.details.items() if key != 'window_sequences')})
table4_v2 = pd.DataFrame(table4_rows).set_index('model').sort_values('tokens_1024')
table4_v2.to_csv(MEASUREMENT_DIR / 'nb27_table4_tokens_v2.csv')
table4_v2

,published_default,published_dynamic_max,tokens_448,tokens_1024,tokens_max,max_at,encoder_tokens_1024,encoder_passes_1024,details_1024
model,,,,,,,,,
CLIP-ViT-L/14,1.0,1.0,1,1,1,112x448,257,1,
SigLIP-L/16,1.0,1.0,1,1,1,112x448,576,1,
SigLIP2-So400m/14,1.0,1.0,1,1,1,112x448,729,1,
BLIP-2-OPT-2.7B,32.0,32.0,32,32,32,112x448,257,1,
InstructBLIP-Vicuna-7B,32.0,32.0,32,32,32,112x448,257,1,
IDEFICS-9B,64.0,64.0,64,64,64,112x448,257,1,
ViLT-B/32,144.0,144.0,145,145,193,336x448,0,0,resized_to=384x384
Gemma-4-26B-A4B,NaN,NaN,258,258,268,336x448,2304,1,resized_to=768x768; soft_token_budget=280
Gemma-4-12B,NaN,NaN,258,258,268,336x448,0,0,resized_to=768x768; soft_token_budget=280


**Reading.** Seven published Table 4 rows do not describe the released configuration: **LLaVA-NeXT** never produces 576 tokens (2,928 for a square image: AnyRes base view + 2 × 2 grid + row newlines); **Idefics2** ships with image splitting on (320 tokens, not 64); **Molmo** pools 2 × 2 patches (144 per crop) and multi-crops (972 at 1024²); **DeepSeek-VL2** compresses each tile to 196 tokens and tiles dynamically (2,017 at 1024²); **BLIP-large**'s captioning decoder cross-attends all 577 image tokens (it is not a pooled-embedding model); **Llama-3.2-Vision** reads up to 4 tiles (6,404 tokens); **Qwen2-VL / Qwen2.5-VL** allow up to 16,384 tokens (max_pixels), not 1,024.

---
## 2. Validation against an op-level FLOP counter

Each transformers-native architecture is rebuilt with reduced depth (random bf16 weights, eager attention), fed the released processor's output, and run once under `torch.utils.flop_counter.FlopCounterMode`. The tensor-level model is evaluated on the same reduced layout and token account. Molmo and DeepSeek-VL2 ship remote code and are not run here; they use the same rules (validated at full depth in notebook 31). Gemma 4 needs transformers 5 and is validated in notebook 33, whose rows are appended below.

In [3]:
validation_rows, head_rows = [], []
for name in VALIDATION_SPECS:
    start = time.time()
    result = validate_model(name)
    validation_rows.extend(result['rows'])
    head_rows.append({'model': name, 'measured_head_gflops': result['measured_head_gflops'],
                      'predicted_head_gflops': result['predicted_head_gflops'], 'seconds': round(time.time() - start)})
validation = pd.DataFrame(validation_rows)
TRANSFORMERS5_VALIDATION = MEASUREMENT_DIR / 'nb33_flop_counter_validation.csv'   # Gemma 4, notebook 33
if TRANSFORMERS5_VALIDATION.exists():
    validation = pd.concat([validation, pd.read_csv(TRANSFORMERS5_VALIDATION).drop(columns='environment')],
                           ignore_index=True)
else:
    print('notebook 33 outputs not found: the Gemma 4 rows are missing from this table')
validation.to_csv(MEASUREMENT_DIR / 'nb27_flop_counter_validation.csv', index=False)
print(f"{len(validation)} component comparisons over {validation['model'].nunique()} models; "
      f"max |relative difference| = {validation['relative_difference_pct'].abs().max():.3f}%")
validation.round(3)

58 component comparisons over 21 models; max |relative difference| = 0.008%


,model,component,measured_gflops,predicted_gflops,relative_difference_pct
0,CLIP-ViT-L/14,vision,13.785,13.785,0.000
1,CLIP-ViT-L/14,fusion,0.003,0.003,0.000
2,CLIP-ViT-L/14,language,0.227,0.227,0.000
3,SigLIP-L/16,vision,35.054,35.054,0.000
4,SigLIP-L/16,language,3.257,3.257,0.000
5,SigLIP2-So400m/14,vision,54.179,54.179,0.000
6,SigLIP2-So400m/14,language,3.938,3.938,0.000
7,BLIP-large,vision,32.675,32.675,-0.000
8,BLIP-large,language,3.923,3.923,-0.000
9,ViLT-B/32,vision,0.679,0.679,0.000


LM heads are listed separately: several forward signatures compute logits at every position, while the cost model charges only the position that yields the first generated token (Section 4.3 below).

In [4]:
pd.DataFrame(head_rows).round(3)

,model,measured_head_gflops,predicted_head_gflops,seconds
0,CLIP-ViT-L/14,0.000,0.000,6
1,SigLIP-L/16,0.000,0.000,7
2,SigLIP2-So400m/14,0.000,0.000,16
3,BLIP-large,0.385,0.048,6
4,ViLT-B/32,0.013,0.013,1
5,BLIP-2-OPT-2.7B,10.302,0.258,14
6,InstructBLIP-Vicuna-7B,10.507,0.263,21
7,LLaVA-1.5-7B,0.263,0.263,16
8,LLaVA-NeXT-7B,0.263,0.263,30
9,IDEFICS-9B,2.884,0.262,35


---
## 3. Revised Table 3

In [5]:
CHAMELEON_VQ_GFLOPS = 554.252107776   # FlopCounterMode on the released VQ-VAE encoder at 512 x 512 (Section 2 run)

def table3(size_accounts):
    """Revised Table 3 rows for one set of token accounts."""
    rows = {}
    for name in size_accounts:
        extra = {'measured_encoder_gflops': CHAMELEON_VQ_GFLOPS} if name == 'Chameleon-7B' else {}
        rows[name] = forward_cost_revised(name, parameter_shapes[name], size_accounts[name], **extra).to_row()
    return pd.DataFrame(rows).T

table3_1024 = table3(accounts[REFERENCE_IMAGE])
table3_448 = table3(accounts[SMALL_IMAGE])
PUBLISHED_TABLE3 = {  # total G, C_V%, vision-attributable %
    'CLIP-ViT-L/14': (170.2, 95.4, 95.4), 'SigLIP-L/16': (418.6, 94.8, 94.8), 'SigLIP2-So400m/14': (735.9, 93.8, 93.8),
    'BLIP-large': (392.4, 97.7, 97.7), 'BLIP-2-OPT-2.7B': (869.1, 60.0, 80.4), 'InstructBLIP-Vicuna-7B': (1398.3, 37.3, 69.1),
    'LLaVA-1.5-7B': (8635.8, 4.4, 95.0), 'LLaVA-NeXT-7B': (9247.8, 4.1, 95.0), 'Molmo-7B-D': (9909.6, 5.0, 95.0),
    'IDEFICS-9B': (809.2, 41.4, 47.7), 'Llama-3.2-11B-Vision': (4849.6, 67.8, 87.8), 'Idefics2-8B': (2140.5, 31.5, 78.7),
    'ViLT-B/32': (31.1, 0.0, 81.8), 'Qwen2-VL-7B': (5684.5, 27.4, 92.0), 'Qwen2.5-VL-7B': (5686.2, 27.4, 92.0),
    'InternVL3-8B': (5158.9, 14.1, 90.5), 'Pixtral-12B': (114749.1, 4.3, 99.3), 'DeepSeek-VL2-Small': (4430.4, 15.6, 96.5),
    'Chameleon-7B': (14892.7, 0.0, 97.0), 'Janus-Pro-7B': (9005.8, 4.4, 95.0)}
published = pd.DataFrame(PUBLISHED_TABLE3, index=['published_total_g', 'published_C_V_pct', 'published_vision_attr_pct']).T
for label, frame in (('1024', table3_1024), ('448', table3_448)):
    frame.join(published).to_csv(MEASUREMENT_DIR / f'nb27_table3_v2_{label}.csv')
summary = pd.DataFrame({
    'published_total_G': published['published_total_g'], 'total_G_448': table3_448['total_g'], 'total_G_1024': table3_1024['total_g'],
    'published_C_V%': published['published_C_V_pct'], 'C_V%_448': table3_448['vision_compute_share_pct'],
    'C_V%_1024': table3_1024['vision_compute_share_pct'], 'vision_attr%_1024': table3_1024['vision_attributable_pct'],
}).sort_values('total_G_1024')
summary.round(1)

,published_total_G,total_G_448,total_G_1024,published_C_V%,C_V%_448,C_V%_1024,vision_attr%_1024
ViLT-B/32,31.1,31.9,31.9,0.0,2.1,2.1,82.3
CLIP-ViT-L/14,170.2,167.5,167.5,95.4,96.7,96.7,96.7
BLIP-large,392.4,411.0,411.0,97.7,93.0,93.0,98.4
SigLIP-L/16,418.6,422.9,422.9,94.8,90.8,90.8,90.8
SigLIP2-So400m/14,735.9,723.5,723.5,93.8,92.7,92.7,92.7
BLIP-2-OPT-2.7B,869.1,857.3,857.3,60.0,60.7,60.7,81.1
IDEFICS-9B,809.2,876.3,876.3,41.4,38.2,38.2,42.7
InstructBLIP-Vicuna-7B,1398.3,1370.6,1370.6,37.3,38.0,38.0,69.7
Gemma-4-26B-A4B,NaN,5015.5,5015.5,NaN,63.3,63.3,95.9
Gemma-4-12B,NaN,6422.3,6422.3,NaN,0.2,0.2,89.0


### How much of the change is method, and how much is the token convention?

The revised cost model is re-run with the **published** token choices. The difference to the published total isolates the accounting fixes (embedding tables and LM heads no longer multiplied by every token, real cross-attention K/V widths, fusion input-side work, MoE activation, per-tile attention, BLIP-2's 39 encoder layers, Chameleon's VQ tokenizer, BLIP-large's cross-attention).

In [6]:
def published_account(name):
    """Token account reproducing the published Table 3 / Table 4 choices."""
    base = dict(model=name, image_height=0, image_width=0)
    spec = {  # encoder sequences, patches, fusion in, fusion out, LM visual tokens, cross-attended
        'CLIP-ViT-L/14': ([257], 256, 0, 1, 0, 0), 'SigLIP-L/16': ([576], 576, 0, 1, 0, 0), 'SigLIP2-So400m/14': ([729], 729, 0, 1, 0, 0),
        'BLIP-large': ([577], 576, 0, 577, 0, 577), 'BLIP-2-OPT-2.7B': ([257], 256, 257, 32, 32, 0),
        'InstructBLIP-Vicuna-7B': ([257], 256, 257, 32, 32, 0), 'LLaVA-1.5-7B': ([577], 576, 576, 576, 576, 0),
        'LLaVA-NeXT-7B': ([577], 576, 576, 576, 576, 0), 'Molmo-7B-D': ([577], 576, 576, 144, 576, 0),
        'IDEFICS-9B': ([257], 256, 257, 64, 0, 64), 'Llama-3.2-11B-Vision': ([1601], 1600, 1601, 1601, 0, 1601),
        'Idefics2-8B': ([729], 729, 729, 64, 64, 0), 'ViLT-B/32': ([], 144, 0, 145, 145, 0),
        'Qwen2-VL-7B': ([1024], 1024, 1024, 256, 256, 0), 'Qwen2.5-VL-7B': ([1024], 1024, 1024, 256, 256, 0),
        'InternVL3-8B': ([1025], 1024, 256, 256, 256, 0), 'Pixtral-12B': ([4096], 4096, 4096, 4096, 4096, 0),
        'DeepSeek-VL2-Small': ([729], 729, 196, 196, 729, 0), 'Chameleon-7B': ([], 0, 0, 1024, 1024, 0),
        'Janus-Pro-7B': ([576], 576, 576, 576, 576, 0)}[name]
    account = TokenAccount(**base, encoder_sequences=spec[0], patch_positions=spec[1], fusion_input_tokens=spec[2],
                           fusion_output_tokens=spec[3], language_visual_tokens=spec[4], cross_attended_tokens=spec[5])
    if name == 'Qwen2.5-VL-7B':
        from vlm_audit.token_accounting import qwen25_window_sequences
        account.details['window_sequences'] = qwen25_window_sequences(32, 32)
    return account

PUBLISHED_ROSTER = list(PUBLISHED_TABLE3)   # the 20 models of the submitted Table 3
table3_method_only = table3({name: published_account(name) for name in PUBLISHED_ROSTER})
decomposition = pd.DataFrame({
    'published_total_G': published['published_total_g'],
    'method_only_total_G': table3_method_only['total_g'],
    'method_effect_%': 100 * (table3_method_only['total_g'] / published['published_total_g'] - 1),
    'revised_1024_total_G': table3_1024.loc[PUBLISHED_ROSTER, 'total_g'],
    'token_effect_x': table3_1024.loc[PUBLISHED_ROSTER, 'total_g'] / table3_method_only['total_g'],
    'published_C_V%': published['published_C_V_pct'], 'method_only_C_V%': table3_method_only['vision_compute_share_pct'],
}).sort_values('method_effect_%')
decomposition.to_csv(MEASUREMENT_DIR / 'nb27_table3_method_vs_tokens.csv')
decomposition.round(2)

,published_total_G,method_only_total_G,method_effect_%,revised_1024_total_G,token_effect_x,published_C_V%,method_only_C_V%
Llama-3.2-11B-Vision,4849.6,3923.47,-19.10,20326.29,5.18,67.8,77.62
Molmo-7B-D,9909.6,8486.73,-14.36,17533.51,2.07,5.0,4.31
InternVL3-8B,5158.9,4530.63,-12.18,43932.15,9.70,14.1,15.97
Janus-Pro-7B,9005.8,7970.75,-11.49,7996.24,1.00,4.4,4.79
Qwen2.5-VL-7B,5686.2,5139.89,-9.61,26801.41,5.21,27.4,25.76
Pixtral-12B,114749.1,106414.12,-7.26,108158.95,1.02,4.3,4.66
Qwen2-VL-7B,5684.5,5279.14,-7.13,31043.09,5.88,27.4,27.72
DeepSeek-VL2-Small,4430.4,4244.03,-4.21,17065.16,4.02,15.6,15.70
InstructBLIP-Vicuna-7B,1398.3,1370.56,-1.98,1370.56,1.00,37.3,37.99
LLaVA-1.5-7B,8635.8,8474.97,-1.86,8474.97,1.00,4.4,4.51


---
## 4. Reviewer item A5, point by point

### (i) The 'ten to thirty percent' omission bound — measured

Element-wise work (normalisation, activations, softmax, residual additions, rotary embeddings, bias additions) is counted at the dispatcher level on the same reduced-depth models, with generous per-element costs for fused kernels (LayerNorm 8, softmax 5, GELU 8, SiLU 4). Embedding lookups are memory reads, not arithmetic.

In [7]:
elementwise = pd.DataFrame([measure_elementwise_share(name) for name in VALIDATION_SPECS]).set_index('model')
if (MEASUREMENT_DIR / 'nb33_elementwise_share.csv').exists():   # Gemma 4, notebook 33 (transformers 5)
    elementwise = pd.concat([elementwise, pd.read_csv(MEASUREMENT_DIR / 'nb33_elementwise_share.csv', index_col='model')])
elementwise.to_csv(MEASUREMENT_DIR / 'nb27_elementwise_share.csv')
print(f"Element-wise share of forward FLOPs: {elementwise['elementwise_share_pct'].min():.2f}% to "
      f"{elementwise['elementwise_share_pct'].max():.2f}% (median {elementwise['elementwise_share_pct'].median():.2f}%)")
elementwise.sort_values('elementwise_share_pct').round(3)

Element-wise share of forward FLOPs: 0.07% to 0.91% (median 0.17%)


,matmul_gflops,elementwise_gflops,elementwise_share_pct
model,,,
Qwen2-VL-7B,653.022,0.454,0.070
Gemma-4-12B,274.588,0.191,0.070
LLaVA-1.5-7B,541.656,0.435,0.080
Janus-Pro-7B,573.800,0.473,0.082
Qwen2.5-VL-7B,361.425,0.299,0.083
Pixtral-12B,1339.775,1.153,0.086
InstructBLIP-Vicuna-7B,72.601,0.069,0.095
BLIP-2-OPT-2.7B,52.274,0.064,0.123
Qwen3-VL-8B,263.397,0.343,0.130


### (ii) Grouped-query attention

The score and value products cost `2·n_q·n_k·(h·d_head)` per layer whether keys and values are shared across heads or not: every query head still scores every key. GQA shrinks only the K/V projection matrices, and those are charged by their real shapes. The published `4LN²d` self-attention term was therefore already correct for GQA models; the published **cross-attention** term was not, because it priced the K/V projections of the visual tokens as d × d matrices.

In [8]:
from transformers import AutoConfig
LLM_CONFIG_SOURCE = {'LLaVA-1.5-7B': 'llava-hf/llava-1.5-7b-hf', 'LLaVA-NeXT-7B': 'llava-hf/llava-v1.6-mistral-7b-hf',
    'Idefics2-8B': 'HuggingFaceM4/idefics2-8b', 'Qwen2-VL-7B': 'Qwen/Qwen2-VL-7B-Instruct', 'Qwen2.5-VL-7B': 'Qwen/Qwen2.5-VL-7B-Instruct',
    'InternVL3-8B': 'OpenGVLab/InternVL3-8B-hf', 'Pixtral-12B': 'mistral-community/pixtral-12b',
    'Llama-3.2-11B-Vision': 'unsloth/Llama-3.2-11B-Vision-Instruct', 'Janus-Pro-7B': 'deepseek-community/Janus-Pro-7B',
    'Chameleon-7B': 'FriendliAI/chameleon-7b', 'InstructBLIP-Vicuna-7B': 'Salesforce/instructblip-vicuna-7b',
    'IDEFICS-9B': 'HuggingFaceM4/idefics-9b', 'BLIP-2-OPT-2.7B': 'Salesforce/blip2-opt-2.7b',
    'Qwen3-VL-8B': 'Qwen/Qwen3-VL-8B-Instruct'}
gqa_rows = []
for name, repository in LLM_CONFIG_SOURCE.items():
    configuration = AutoConfig.from_pretrained(repository)
    text = getattr(configuration, 'text_config', configuration)
    heads = getattr(text, 'num_attention_heads', None)
    gqa_rows.append({'model': name, 'attention_heads': heads, 'key_value_heads': getattr(text, 'num_key_value_heads', heads) or heads})
gqa_rows += [{'model': 'Molmo-7B-D', 'attention_heads': 28, 'key_value_heads': 4},
             {'model': 'DeepSeek-VL2-Small', 'attention_heads': 16, 'key_value_heads': 'latent (rank 512)'},
             # Gemma 4 configs need transformers 5: values from the released config.json (global layers reuse keys as values).
             {'model': 'Gemma-4-12B', 'attention_heads': 16, 'key_value_heads': '8 (sliding) / 1 (global)'},
             {'model': 'Gemma-4-26B-A4B', 'attention_heads': 16, 'key_value_heads': '8 (sliding) / 2 (global)'}]
gqa = pd.DataFrame(gqa_rows).set_index('model')
gqa.to_csv(MEASUREMENT_DIR / 'nb27_attention_heads.csv')
gqa

,attention_heads,key_value_heads
model,,
LLaVA-1.5-7B,32,32
LLaVA-NeXT-7B,32,8
Idefics2-8B,32,8
Qwen2-VL-7B,28,4
Qwen2.5-VL-7B,28,4
InternVL3-8B,28,4
Pixtral-12B,32,8
Llama-3.2-11B-Vision,32,8
Janus-Pro-7B,32,32


In [9]:
# Cross-attention visual term: published d x d convention vs real K/V shapes, at the published token counts.
def published_cross_kv(layers, tokens, width):
    """Published K/V projection term: 2 x (2 d^2 N_v) per layer."""
    return layers * 2 * (2 * width * width * tokens) / 1e9
for name, kv_input, kv_output in (('Llama-3.2-11B-Vision', 4096, 1024), ('IDEFICS-9B', 1280, 4096)):
    account = published_account(name)
    real = 8 * 2 * (2 * kv_input * kv_output * account.cross_attended_tokens) / 1e9
    print(f"{name}: K/V projections of {account.cross_attended_tokens} visual tokens over 8 layers: published convention "
          f"{published_cross_kv(8, account.cross_attended_tokens, 4096):.1f} G, real shapes ({kv_input}->{kv_output}) {real:.1f} G")

Llama-3.2-11B-Vision: K/V projections of 1601 visual tokens over 8 layers: published convention 859.5 G, real shapes (4096->1024) 214.9 G
IDEFICS-9B: K/V projections of 64 visual tokens over 8 layers: published convention 34.4 G, real shapes (1280->4096) 10.7 G


### (iii) LM head and embeddings

The published protocol multiplied the embedding table by every token for all models (`2PN` with P including `embed_tokens`) and the LM head as well for InternVL3, Molmo, DeepSeek-VL2 and Janus-Pro. Neither is a per-token matmul in prefill: the embedding is a lookup, and producing the first output token needs the head at one position. The revised model charges the head once for every model:

In [10]:
table3_1024[['head_g']].T.round(2)

,CLIP-ViT-L/14,SigLIP-L/16,SigLIP2-So400m/14,ViLT-B/32,BLIP-large,BLIP-2-OPT-2.7B,InstructBLIP-Vicuna-7B,LLaVA-1.5-7B,LLaVA-NeXT-7B,Molmo-7B-D,IDEFICS-9B,Llama-3.2-11B-Vision,Idefics2-8B,Qwen2-VL-7B,Qwen2.5-VL-7B,InternVL3-8B,Pixtral-12B,DeepSeek-VL2-Small,Chameleon-7B,Janus-Pro-7B,Qwen3-VL-8B,Gemma-4-12B,Gemma-4-26B-A4B
head_g,0.0,0.0,0.0,0.01,0.05,0.26,0.26,0.26,0.26,1.09,0.26,1.05,0.26,1.09,1.09,1.09,1.34,0.42,0.54,0.84,1.24,2.01,1.48


### (iv) BLIP-2 / InstructBLIP encoder depth

Layer counts are now read from the parameter layout itself, not typed in.

In [11]:
from vlm_audit.tensor_cost_model import count_layers
for name in ('BLIP-2-OPT-2.7B', 'InstructBLIP-Vicuna-7B', 'Molmo-7B-D', 'Llama-3.2-11B-Vision'):
    print(name, count_layers(name, parameter_shapes[name].keys()))

BLIP-2-OPT-2.7B {'vision': 39, 'qformer': 12, 'qformer_cross': 6, 'language': 32}
InstructBLIP-Vicuna-7B {'vision': 39, 'qformer': 12, 'qformer_cross': 6, 'language': 32}
Molmo-7B-D {'vision': 23, 'language': 28}
Llama-3.2-11B-Vision {'vision': 32, 'vision_global': 8, 'language': 32, 'cross': 8}


---
## 5. ViLT (A7) and the V%–C_V% relationship (revised Table 7)

ViLT's patch projection is now costed (a 32 × 32 × 3 → 768 projection over 144 patches), and its V% uses the corrected attribution of notebook 26 (image-only embeddings, 2.1%).

In [12]:
SEPARABLE = [name for name in TABLE_ORDER if name != 'Chameleon-7B']

def correlation(compute_table, names, label):
    """Spearman rho(V%, C_V%), permutation p, 95% Fisher interval."""
    vision_share = components.loc[names, 'V_pct'].astype(float)
    compute_share = compute_table.loc[names, 'vision_compute_share_pct'].astype(float)
    test = spearman_permutation_test(vision_share.values, compute_share.values)
    low, high = fisher_interval(test['rho'], test['sample_size'])
    return {'analysis': label, 'n': test['sample_size'], 'rho': test['rho'], 'p_perm': test['p_value'],
            'exact': test['exact'], 'ci_low': low, 'ci_high': high}

without_vilt = [name for name in SEPARABLE if name != 'ViLT-B/32']
correlations = pd.DataFrame([
    correlation(table3_1024, SEPARABLE, 'revised, 1024 x 1024 input'),
    correlation(table3_1024, without_vilt, 'revised, 1024 x 1024 input, without ViLT'),
    correlation(table3_448, SEPARABLE, 'revised, 448 x 448 input'),
    correlation(table3_448, without_vilt, 'revised, 448 x 448 input, without ViLT'),
    correlation(table3_method_only, [name for name in SEPARABLE if name in PUBLISHED_ROSTER],
                'revised method, published token choices (20 submitted models)'),
    correlation(table3_1024, [name for name in SEPARABLE if name in PUBLISHED_ROSTER],
                'revised, 1024 x 1024 input, 20 submitted models only'),
]).set_index('analysis')
correlations.to_csv(MEASUREMENT_DIR / 'nb27_vision_share_correlation.csv')
correlations.round(3)

,n,rho,p_perm,exact,ci_low,ci_high
analysis,,,,,,
"revised, 1024 x 1024 input",22,0.718,0.000,False,0.425,0.875
"revised, 1024 x 1024 input, without ViLT",21,0.675,0.001,False,0.344,0.857
"revised, 448 x 448 input",22,0.743,0.000,False,0.467,0.887
"revised, 448 x 448 input, without ViLT",21,0.704,0.001,False,0.391,0.871
"revised method, published token choices (20 submitted models)",19,0.902,0.000,False,0.758,0.962
"revised, 1024 x 1024 input, 20 submitted models only",19,0.775,0.000,False,0.496,0.909


In [13]:
table7_v2 = pd.DataFrame({
    'V_pct': components.loc[SEPARABLE, 'V_pct'].astype(float),
    'C_V_pct_1024': table3_1024.loc[SEPARABLE, 'vision_compute_share_pct'].astype(float),
    'C_V_pct_448': table3_448.loc[SEPARABLE, 'vision_compute_share_pct'].astype(float),
})
table7_v2['ratio_1024'] = table7_v2['C_V_pct_1024'] / table7_v2['V_pct']
table7_v2['ratio_448'] = table7_v2['C_V_pct_448'] / table7_v2['V_pct']
table7_v2 = table7_v2.sort_values('ratio_1024', ascending=False)
table7_v2.to_csv(MEASUREMENT_DIR / 'nb27_table7_v2.csv')
table7_v2.round(2)

,V_pct,C_V_pct_1024,C_V_pct_448,ratio_1024,ratio_448
Gemma-4-26B-A4B,2.21,63.32,63.32,28.69,28.69
DeepSeek-VL2-Small,2.65,39.05,39.68,14.73,14.96
Idefics2-8B,4.97,62.14,36.68,12.50,7.38
Llama-3.2-11B-Vision,8.09,90.92,77.62,11.23,9.59
Molmo-7B-D,3.63,20.88,18.03,5.76,4.97
Qwen3-VL-8B,4.74,25.85,18.03,5.45,3.80
IDEFICS-9B,7.06,38.18,38.18,5.41,5.41
Qwen2-VL-7B,7.61,38.08,27.58,5.00,3.62
InternVL3-8B,3.83,16.47,15.88,4.30,4.15
Qwen2.5-VL-7B,7.62,28.28,25.63,3.71,3.36


---
## 6. A9 in one line: LLaVA-1.5 versus LLaVA-NeXT

In [14]:
for label, frame in (('published (576 tokens each)', None), ('revised, 448 x 448', table3_448), ('revised, 1024 x 1024', table3_1024)):
    if frame is None:
        one, next_ = PUBLISHED_TABLE3['LLaVA-1.5-7B'][0], PUBLISHED_TABLE3['LLaVA-NeXT-7B'][0]
    else:
        one, next_ = frame.loc['LLaVA-1.5-7B', 'total_g'], frame.loc['LLaVA-NeXT-7B', 'total_g']
    print(f'{label:32s} LLaVA-1.5 {one:9.0f} G   LLaVA-NeXT {next_:9.0f} G   ratio {next_ / one:4.2f}')

published (576 tokens each)      LLaVA-1.5      8636 G   LLaVA-NeXT      9248 G   ratio 1.07
revised, 448 x 448               LLaVA-1.5      8475 G   LLaVA-NeXT     47942 G   ratio 5.66
revised, 1024 x 1024             LLaVA-1.5      8475 G   LLaVA-NeXT     47942 G   ratio 5.66
